# News classifier transfor leaning from LLM 新聞類別分類器(遷移學習)

This is a comprahensive notebook and tutorial on how to fine tune the `qwen-0.5b` classification model

Fine-tuning Qwen-0.5B (a smaller model) with LoRA (Low-Rank Adaptation) is an efficient approach that requires less computational power.

accuracy = 0.90



# Transformers變形金剛(變壓器)模型

<img src="https://cdn.analyticsvidhya.com/wp-content/uploads/2019/06/Screenshot-from-2019-06-17-19-53-10.png">

Bert and GPT are parts of Transformer

<img src="https://heidloff.net/assets/img/2023/02/transformers.png" width="800">

Encoders and Decoders
As mentioned, there are encoders and decoders. BERT uses encoders only, GTP uses decoders only. Both options understand language including syntax and semantics. Especially the next generation of large language models like GPT with billions of parameters do this very well.

The two models focus on different scenarios. However, since the field of foundation models is evolving, the differentiation is often fuzzier.

BERT (encoder): classification (e.g., sentiment), questions and answers, summarization, named entity recognition
GPT (decoder): translation, generation (e.g., stories)
The outputs of the core models are different:

BERT (encoder): Embeddings representing words with attention information in a certain context
GPT (decoder): Next words with probabilities
Both models are pretrained and can be reused without intensive training. Some of them are available as open source and can be downloaded from communities like Hugging Face, others are commercial. Reuse is important, since trainings are often very resource intensive and expensive which few companies can afford.

The pretrained models can be extended and customized for different domains and specific tasks. Layers can sometimes be reused without modifications and more layers are added on top. If layers need to be modified, the new training is more expensive. The technique to customize these models is called Transfer Learning, since the same generic model can easily be transferred to other domains.

[Source](https://heidloff.net/article/foundation-models-transformers-bert-and-gpt/)

### Encoder-Decoder Transformers



<img src='https://miro.medium.com/v2/resize:fit:1400/format:webp/1*vrSX_Ku3EmGPyqF_E-2_Vg.png'>

### Attension Layer
<img src='https://miro.medium.com/v2/resize:fit:640/format:webp/1*aTLu4HxVUzEOIZTLHmOktw.png'>

## MLP GPT Animation

MLP Forward and Backward Pass:

https://www.youtube.com/watch?v=sLsCN9ZL9RI

Large Language Models explained briefly:

https://www.youtube.com/watch?v=LPZh9BOjkQs

# Colab GPU
## 設定執行階段 變更執行階段類型 硬體加速器  選取GPU  


In [1]:
!nvidia-smi

Tue May 27 11:28:10 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 546.80                 Driver Version: 546.80       CUDA Version: 12.3     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                     TCC/WDDM  | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 3060 ...  WDDM  | 00000000:01:00.0 Off |                  N/A |
| N/A   56C    P5              19W /  85W |    553MiB /  6144MiB |     17%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [2]:
# 列出GPU與CPU資源
from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())

[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 10155094432329239946
xla_global_id: -1
]


In [3]:
# Specify the GPU device to use
#import os
#os.environ["CUDA_VISIBLE_DEVICES"] = "0"

## Ubuntu版本資訊

In [4]:
!lsb_release -a

'lsb_release' ���O�����Υ~���R�O�B�i���檺�{���Χ妸�ɡC


# Insatll packages


        transformers: For model handling.
        peft: For LoRA integration.
        datasets: For dataset handling.
        accelerate: For distributed training.
        bitsandbytes: For memory-efficient training (optional but useful for Qwen).

In [5]:
#!pip install transformers peft datasets accelerate bitsandbytes

In [6]:
!pip install evaluate

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
anaconda-cloud-auth 0.1.3 requires pydantic<2.0, but you have pydantic 2.11.4 which is incompatible.
prophet 1.1.5 requires holidays>=0.25, but you have holidays 0.10.3 which is incompatible.



  Using cached requests-2.32.3-py3-none-any.whl.metadata (4.6 kB)
   ---------------------------------------- 0.0/25.8 MB ? eta -:--:--
   ---- ----------------------------------- 2.6/25.8 MB 11.6 MB/s eta 0:00:02
   -------- ------------------------------- 5.2/25.8 MB 11.8 MB/s eta 0:00:02
   ----------- ---------------------------- 7.6/25.8 MB 11.7 MB/s eta 0:00:02
   --------------- ------------------------ 10.2/25.8 MB 11.8 MB/s eta 0:00:02
   ------------------ --------------------- 12.1/25.8 MB 12.0 MB/s eta 0:00:02
   ---------------------- ----------------- 14.7/25.8 MB 11.8 MB/s eta 0:00:01
   --------------------------- ------------ 17.6/25.8 MB 11.7 MB/s eta 0:00:01
   ------------------------------- -------- 20.2/25.8 MB 11.7 MB/s eta 0:00:01
   ---------------------------------- ----- 22.3/25.8 MB 11.7 MB/s eta 0:00:01
   -------------------------------------- - 24.9/25.8 MB 11.7 MB/s eta 0:00:01
   ---------------------------------------- 25.8/25.8 MB 11.4 MB/s eta 0:00:

# 掛載雲端硬碟

In [7]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

# 切換工作目錄到雲端硬碟目錄下(取用資料比較方便，可以用相對路徑)

你要改成你自己雲端硬碟目錄的路徑，可複製路徑並貼上，不要用鍵盤輸入!

In [ ]:
# cd to_your_folder
# cd命令的前面一行不要加上說明文字，否則colab的cd會認不得指令

In [ ]:
cd /content/drive/MyDrive/Colab Notebooks

In [ ]:
ls -l

In [ ]:
import torch
import datasets
import pandas as pd
import evaluate
import numpy as np

# Load Huggingface transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM
from sklearn import metrics
import torch
import evaluate
import datasets


In [ ]:
# Setting device on GPU if available, else CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


# Read data

中央社新聞2.6萬篇

In [ ]:
file_name= 'news_dataset_preprocessed_for_django.csv'
df = pd.read_csv(file_name, sep='|')

In [ ]:
df.head(2)

,item_id,title,category,content,link,date,photo_link,tokens_v2,top_keys_freq,summary,sentiment
0,aipl_20220124_1001,殲16D首擾台 專家示警：國軍應強化電子作戰,政治,中共殲16D新型電戰機今天首次擾台，學者及退將分析，殲16D可能已量產並進行實戰化運用，國軍...,https://www.cna.com.tw/news/aipl/202201240357....,2022-01-24,https://imgcdn.cna.com.tw/www/WebPhotos/200/20...,"['中共', '新型', '電戰機', '學者', '退將', '分析', '量產', '進...","[('中共', 7), ('台灣', 7), ('電子', 6), ('美軍', 6), (...","['其中殲16D新型電戰機為首次現蹤', '中共殲16D新型電戰機今天首次擾台', '13架...",0.96
1,aipl_20220124_1002,朝野協商總預算在促轉會卡關 25日拚三讀添變數,政治,立法院朝野黨團今天繼續協商總預算案，不過到促轉會的預算提案時，朝野為了蔣經國總統圖書館議題起...,https://www.cna.com.tw/news/aipl/202201240390....,2022-01-24,NaN,"['立法院', '朝野', '黨團', '總預算案', '促轉會', '預算', '提案',...","[('預算', 7), ('提案', 7), ('黨團', 6), ('總統', 6), (...","['民進黨立法院黨團總召柯建銘說', '促轉會說這叫威權', '不過到促轉會的預算提案時',...",0.74


In [ ]:
df.groupby('category').size()

category
兩岸    1555
國際    6228
娛樂     639
政治    2988
文化     666
生活    5920
產經    2541
社會    2055
科技     171
證卷    1793
運動    1623
dtype: int64

In [ ]:
df.dtypes

item_id           object
title             object
category          object
content           object
link              object
date              object
photo_link        object
tokens_v2         object
top_keys_freq     object
summary           object
sentiment        float64
dtype: object

## Select and rename column fields

In [ ]:
df = df[['content','category']]

In [ ]:
df = df.rename(columns={'content': 'text'}) 

In [ ]:
df

,text,category
0,中共殲16D新型電戰機今天首次擾台，學者及退將分析，殲16D可能已量產並進行實戰化運用，國軍...,政治
1,立法院朝野黨團今天繼續協商總預算案，不過到促轉會的預算提案時，朝野為了蔣經國總統圖書館議題起...,政治
2,立法院今天三讀修正通過加重酒駕刑罰及行政罰的相關條文，國民黨主席朱立倫表示，酒駕零容忍，民進...,政治
3,副總統賴清德明天將展開任內首次出訪，以總統特使身分，出席27日宏都拉斯新任總統卡斯楚就職典禮...,政治
4,台灣獨立建國聯盟今天指出，蔣經國「反共」，更是「獨裁者」，蔣經國「保台」實則「保蔣」，盼總統...,政治
...,...,...
26174,記錄上海因疫封城下各種次生災害的短片「四月之聲」，近日在中國網路熱傳，隨之而來的是網管的審查...,兩岸
26175,上海民眾仰賴的保供物資連接出包，繼上海市紀監委表示要嚴查背後是否涉及收受回扣及利益輸送行為後...,兩岸
26176,疫情之下「物資援滬」成中國各省區的「全民運動」。但陸續傳出部分物資品質有問題，造成上海與其他...,兩岸
26177,中國本輪COVID-19本土病例大部分來自上海市，隨著當地疫情反覆，近日略降的病例數再次突破...,兩岸


# Convert the format of y 

Convert the format of y from int to LongTensor

## Convert label using one-hot representation 輸出資料格式one-hot轉換
    
    轉成用2個節點表達兩類
    類別0: [1 0]  
    類別1: [0 1]  


    如果是3個類別用3個節點表之:
    類別0: [1 0 0]  
    類別1: [0 1 0]  
    類別2: [0 0 1]

負面情緒Negative 0 --> [1 0] 

正面情緒Positive 1 --> [0 1] 

## Easy Represention 0,1,2... (內部會自動轉換為one-hot)

    負面情緒Negative 0 --> [0] 

    正面情緒Positive 1 --> [1] 

    如果是3個類別
    中立情緒 Neutral 2 --> [2] 


In [ ]:
# Map labels to integers
categories=['政治','科技','運動','證卷','產經','娛樂','生活','國際','社會','文化','兩岸']

In [ ]:

label_to_id = { cate : i for i, cate in enumerate(categories)}

In [ ]:
label_to_id

{'政治': 0,
 '科技': 1,
 '運動': 2,
 '證卷': 3,
 '產經': 4,
 '娛樂': 5,
 '生活': 6,
 '國際': 7,
 '社會': 8,
 '文化': 9,
 '兩岸': 10}

In [ ]:
id_to_label = { i : cate for i, cate in enumerate(categories)}

In [ ]:
id_to_label

{0: '政治',
 1: '科技',
 2: '運動',
 3: '證卷',
 4: '產經',
 5: '娛樂',
 6: '生活',
 7: '國際',
 8: '社會',
 9: '文化',
 10: '兩岸'}

## Label comlumn with numeric representation

In [ ]:
def encode_labels(example):
    example["label"] = label_to_id[example["category"]]
    return example

df = df.apply(encode_labels, axis=1)

In [ ]:
df.head(1)

,text,category,label
0,中共殲16D新型電戰機今天首次擾台，學者及退將分析，殲16D可能已量產並進行實戰化運用，國軍...,政治,0


In [ ]:
df = df[['text','label']]

In [ ]:
df.head(1)

,text,label
0,中共殲16D新型電戰機今天首次擾台，學者及退將分析，殲16D可能已量產並進行實戰化運用，國軍...,0


# Sample some examples for demonstration

In [ ]:
# df = df.sample(5000)

# Conver pandas dataframe to Huggingface Dataset

In [ ]:
dataset = datasets.Dataset.from_pandas(df, preserve_index=False)

In [ ]:
dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 26179
})

In [ ]:
dataset[0]

{'text': '中共殲16D新型電戰機今天首次擾台，學者及退將分析，殲16D可能已量產並進行實戰化運用，國軍應強化電子作戰，並與美日分享電子參數，畢竟誰掌握制電磁權誰就握有戰場主動權。空軍晚間6時20分發布共機動態，13架共機（8架殲16戰機、1架運8反潛機、2架轟6、2架殲16D電戰機）侵擾台灣西南防空識別區，其中殲16D新型電戰機為首次現蹤，另外，1架運8反潛機深入台灣南方及東南方空域。熟悉中共軍事的中山大學亞太事務英語學程兼任助理教授林穎佑接受中央社訪問時指出，殲16D今天擾台時有無開啟電戰裝備外界無從得知，但可以觀察，昨天擾台的39架共機中並未出現空警500，可能是中共擔憂參數遭正在沖繩南方海域演習的美軍反偵蒐，但藉由今天的殲16D，以演練干擾及刺探美軍。針對今天的共機態勢，林穎佑則認為是進攻型編隊。他解釋，運8反潛機今天深入台灣南方及東南方海域，目的應是偵蒐在巴士海峽一帶的美軍或他國潛艦，而殲16戰機則是掩護轟6及殲16D。林穎佑表示，國軍對於殲16D的反制能力外界仍然無法得知，但未來除了可以關注共軍擾台的飛行路徑、機種外，更重要的是國軍應強化看不見的電子作戰的戰場，「因為在未來誰能掌握制電磁權，誰就能影響戰爭勝負」。退役空軍副司令張延廷中將告訴中央社記者，殲16D是中共的王牌，與美軍電戰機能力旗鼓相當，甚至有部分的性能可能還超越美軍，此次首次現蹤，在台海作戰上分別有「性能」、「實戰化」及「量產」三個指標。張延廷分析，殲16D這次亮相代表中共不怕外界有能力蒐集相關電子參數，並炫耀其強大電戰能力，而中共一直強調要在複雜電磁環境的作戰體系中打仗，未來台海作戰環境中，國軍硬殺武器應強化反干擾能力，畢竟誰掌握制電磁權誰就握有戰場主動權。張延廷也說，此次殲16D現蹤可能代表已經量產且達到一定數量，不僅中共空軍能力已獲強化，更有辦法進行實戰化運用。至於國軍因應方式，張延廷認為，台灣必須全力進行情蒐，並研擬電磁反制、電磁反反制方式。此外，美國、日本應釋出電子參數給台灣，因為美軍近年來時常在廣東、海南、黃海、浙江等沿海抵近飛行，所截獲的電子參數會比台灣來得多，並用於台灣的戰場經營、管理及運用，並結合訓練。',
 'label': 0}

In [ ]:
dataset.to_pandas().head(5)

,text,label
0,中共殲16D新型電戰機今天首次擾台，學者及退將分析，殲16D可能已量產並進行實戰化運用，國軍...,0
1,立法院朝野黨團今天繼續協商總預算案，不過到促轉會的預算提案時，朝野為了蔣經國總統圖書館議題起...,0
2,立法院今天三讀修正通過加重酒駕刑罰及行政罰的相關條文，國民黨主席朱立倫表示，酒駕零容忍，民進...,0
3,副總統賴清德明天將展開任內首次出訪，以總統特使身分，出席27日宏都拉斯新任總統卡斯楚就職典禮...,0
4,台灣獨立建國聯盟今天指出，蔣經國「反共」，更是「獨裁者」，蔣經國「保台」實則「保蔣」，盼總統...,0


# Load Tokenizer

In [ ]:
model_id = "Qwen/Qwen2.5-0.5B-instruct"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)


In [ ]:
# model_id = "google/gemma-3-1b-it" # 2.0 GB

# 必須在Huggingface註冊，取得API token才能下載模型
#access_token = "???"
#tokenizer = AutoTokenizer.from_pretrained(model_id, token=access_token)

# Tokenzie text

In [ ]:
def tokenize_function(example):
    return tokenizer(
        example["text"], 
        #padding="max_length", 
        max_length=512,
        truncation=True, 
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/26179 [00:00<?, ? examples/s]

In [ ]:
tokenized_dataset

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 26179
})

In [ ]:
# tokenized_dataset[0]

# Split dataset for training and testing

# Split dataset: Train, Test (Val) 

Training set: 訓練資料集 -->給模型讀進去訓練

Test set: 測試資料集 -->驗證或測試模型的準確度



Split dataset into 90% for training and 10% for testing

In [ ]:
tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.05, seed=1234)
train_data = tokenized_dataset["train"]
test_data = tokenized_dataset["test"]

In [ ]:
train_data

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 24870
})

In [ ]:
print(test_data)

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 1309
})


In [ ]:
train_data[0]

{'text': '台灣染疫率萬分之17 陳時中：若達15%實質與病毒共存國內21日新增2969例COVID-19本土病例，台北市副市長黃珊珊示警，全台灣未來會有20%人口、約460萬人確診。中央流行疫情指揮中心指揮官陳時中21日回應說，20%人口確診並非不可能，但希望能控制在15到16%。指揮中心估算，目前台灣染疫率約萬分之17（0.17%），若染疫率高達15%至20%，就是實質與病毒共存，現行政策重點在疫苗、藥物與輕重症分流，以減災為目標。（看完整報導）篩檢代替隔離逐步開放 協商讓藥局可供應快篩試劑國內COVID-19疫情持續上升，確診人數不斷創新高，目前規定醫護接種3劑COVID-19疫苗，即使曾與確診者接觸，被匡列須隔離，也能以做快篩或PCR代替。指揮中心預計本週公布擴大篩檢取代隔離政策適用的職業，包括關鍵基礎設施、特殊公務單位等，以確保社會運作。指揮官陳時中21日表示，這個政策將逐步前行，最後走到全民適用。對於外界關切目前藥局仍買不到快篩試劑，陳時中回應，現在便利超商比較容易買得到，指揮中心仍積極協商讓藥局能夠進貨。（看完整報導）居家照護取消同住者健康條件限制 未滿3個月嬰兒發燒住院收治疫情指揮中心21日宣布調整COVID-19確診者分流收治原則，即日起取消居家照護同住者必須小於65歲、無懷孕或洗腎等健康條件限制，並調整居家照護同住者快篩頻率。另外，年齡未滿3個月的嬰兒如果發燒，或年齡為3至12個月嬰兒高燒超過攝氏39度，或須進行血液透析者，均收治於醫院。指揮中心並提醒染疫兒童5大警訊表徵，出現症狀須視訊診療，必要時得安排外出就醫；若現呼吸困難等6大症狀可緊急自行就醫。針對兒童疫苗即將開打，指揮中心表示，莫德納疫苗數量充足，預估到5月可施打500萬人，BNT疫苗採購也在簽約中，若順利首批最快5月可到貨。（看完整報導）俄軍在烏克蘭占領區遍插勝利旗 5/9前恐加強攻勢俄軍占領的烏克蘭各地區已開始出現二戰時期的蘇聯勝利軍旗，英國國防部指出，俄國可能想在5月9日二戰勝利日慶祝活動之前展示重大戰果，而這可能會影響俄軍未來幾天的作戰行動。烏克蘭南部港市馬立波最後一批守軍20日不理會俄方要他們棄械投降的最後通牒，並呼籲外界提供安全擔保，讓受困的平民安全離開。（看完整報導）政府修法放寬就業港人定居條件 預計5月生效為解決來台就業港人只可居留但無法長期定居的問題，行政院大陸

#  定義Model

這裡的做法有簡單的版本也有複雜的版本

簡單版:


        model = AutoModelForSequenceClassification.from_pretrained(
            model_id,  # 模型名稱
            num_labels=len(categories),  # Number of output labels
        )


複雜版:




In [ ]:
from transformers import Qwen2Model, Trainer, TrainingArguments, AutoTokenizer, AutoModel
from torch import nn
from transformers.modeling_outputs import SequenceClassifierOutput
import os

In [ ]:
import torch
import os
import torch.nn.functional as F
from torch import nn

class QwenForClassifier(nn.Module):
    def __init__(self, base_model, hidden_size, num_labels, num_fusion_layers=4):
        super(QwenForClassifier, self).__init__()
        # 指定基礎模型 來自於外部指定的模型
        self.base_model = base_model
        
        # 融合權重多少層 (可調整層數)
        self.num_fusion_layers = num_fusion_layers
        # 保存config配置
        self.config = base_model.config
        self.config.num_labels = num_labels
        
        # 凍結 base model 的參數
        for param in self.base_model.parameters():
            param.requires_grad = False
        
        # 多層融合權重 (最後4層)
        # 初始值是0.25，訓練後模型可能會學到最後一層權重為0.4，倒數第二層為0.3，倒數第三層為0.2，倒數第四層為0.1
        self.layer_weights = nn.Parameter(torch.ones(num_fusion_layers) / num_fusion_layers)
            
        # 新增 QKV 線性層
        self.q_proj = nn.Linear(hidden_size, hidden_size)
        self.k_proj = nn.Linear(hidden_size, hidden_size)
        self.v_proj = nn.Linear(hidden_size, hidden_size)
        
        # 增強型分類器
        # 這是一個三層的神經網絡，將高維特徵映射到類別空間：
        #     1. 第一層：將1024維降到256維
        #     2. 第二層：將256維降到128維
        #     3. 第三層：將128維降到類別數量
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.2),
            
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.1),
            
            nn.Linear(128, num_labels)
        )
        
    
    def forward(self, input_ids, attention_mask=None, labels=None):
        # 獲取所有隱藏層狀態
        outputs = self.base_model(
            input_ids=input_ids, 
            attention_mask=attention_mask,
            output_hidden_states=True # 設置為True以獲取所有隱藏層狀態
        )
        
        # outputs 是一個包含多個元素的元組
        hidden_states = outputs.hidden_states # 有24層的隱藏狀態 維度是
        # 獲取最後4層隱藏狀態
        
        last_layers = hidden_states[-self.num_fusion_layers:] # 取得最後4層的隱藏狀態 維度是 (batch_size, seq_len, hidden_size) 
        layer_weights = F.softmax(self.layer_weights, dim=0) # 將權重轉換為概率分佈 維度是 (num_fusion_layers,)
        # 加權融合最後4層特徵
        sequence_output = torch.zeros_like(last_layers[0]) # 初始化為0，維度是 (batch_size, seq_len, hidden_size)
        for i, layer in enumerate(last_layers):
            sequence_output += layer_weights[i].unsqueeze(-1).unsqueeze(-1) * layer # 將權重擴展到 (batch_size, seq_len, hidden_size)
    
        # ===== QKV Attention Pooling =====
        # [batch, seq_len, hidden_size]
        Q = self.q_proj(sequence_output)
        K = self.k_proj(sequence_output)
        V = self.v_proj(sequence_output)
        # Attention scores: [batch, seq_len, seq_len]
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / (Q.size(-1) ** 0.5)
        attn_probs = F.softmax(attn_scores, dim=-1)
        # Context vector: [batch, seq_len, hidden_size]
        context_vector_qkv = torch.matmul(attn_probs, V).mean(dim=1)  # [batch, hidden_size]

        # ===== mean pooling =====
        mean_pooled = torch.mean(sequence_output, dim=1)

        # ===== 融合 QKV context 與 mean_pooled =====
        combined_repr = context_vector_qkv + mean_pooled

        # 分類預測
        logits = self.classifier(combined_repr) # 維度是 (batch_size, num_labels)
        
        # 計算損失
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss() # 計算交叉熵損失
            loss = loss_fct(logits, labels)
            
        return {"loss": loss, "logits": logits}
    
    def save_model(self, output_dir=None):
        """保存分類器權重和配置"""
        os.makedirs(output_dir, exist_ok=True)
        
        # 保存分類器權重與QKV權重
        classifier_path = os.path.join(output_dir, "classifier_weights.pt")
        model_dict = {
            'classifier': self.classifier.state_dict(),
            'layer_weights': self.layer_weights,
            'q_proj': self.q_proj.state_dict(),
            'k_proj': self.k_proj.state_dict(),
            'v_proj': self.v_proj.state_dict(),
            'config': {
                'num_labels': self.config.num_labels,
                'hidden_size': self.config.hidden_size
            }
        }
        torch.save(model_dict, classifier_path)
        print(f"已保存分類器權重至 {classifier_path}")
    
    def load_model(self, model_dir, device=None):
        """載入分類器權重"""
        if device is None:
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
            
        classifier_path = os.path.join(model_dir, "classifier_weights.pt")
        if os.path.exists(classifier_path):
            model_dict = torch.load(classifier_path, map_location=device, weights_only=True)
            
            # 載入各組件
            self.classifier.load_state_dict(model_dict['classifier'])
            self.layer_weights.data = model_dict['layer_weights'].to(device)
            self.q_proj.load_state_dict(model_dict['q_proj'])
            self.k_proj.load_state_dict(model_dict['k_proj'])
            self.v_proj.load_state_dict(model_dict['v_proj'])
            
            print(f"已載入分類器權重: {classifier_path}")
            return True
        else:
            print(f"警告: 找不到分類器權重檔案 {classifier_path}")
            return False

## 初始化模型


        分類任務：兩者都可用，但 AutoModel 更輕量
        生成功能：只有 AutoModelForCausalLM 支持
        內存使用：AutoModelForCausalLM 通常較大，因為包含了完整的語言模型頭
        在 Qwen2 情感分類模型中，使用 AutoModelForCausalLM 更為靈活，因為它既可以進行分類，也保留了原始的文本生成功能。



        # AutoModelForCausalLM 輸出
        outputs = base_model(input_ids, attention_mask)
        # 輸出包含：
        # - loss: (可選) 語言模型損失
        # - logits: 張量，形狀為 [batch_size, sequence_length, vocab_size]
        # - past_key_values: (可選) 用於加速解碼的過去狀態
        # - hidden_states: (可選) 所有隱藏層狀態的元組
        # - attentions: (可選) 注意力權重的元組


        # AutoModel 輸出
        outputs = base_model(input_ids, attention_mask)
        # 輸出包含：
        # - last_hidden_state: 張量，形狀為 [batch_size, sequence_length, hidden_size]
        # - hidden_states: (可選) 所有隱藏層狀態的元組
        # - attentions: (可選) 注意力權重的元組

In [ ]:
len(categories)

11

In [ ]:
# 在外部先載入base_model預訓練權重(不包含分類層)
full_model = AutoModelForCausalLM.from_pretrained(model_id)
#
hidden_size = full_model.config.hidden_size
model = QwenForClassifier(full_model.model, hidden_size, num_labels= len(categories))

# 移動到指定設備
model = model.to(device)

In [ ]:
full_model.model

Qwen2Model(
  (embed_tokens): Embedding(151936, 896)
  (layers): ModuleList(
    (0-23): 24 x Qwen2DecoderLayer(
      (self_attn): Qwen2SdpaAttention(
        (q_proj): Linear(in_features=896, out_features=896, bias=True)
        (k_proj): Linear(in_features=896, out_features=128, bias=True)
        (v_proj): Linear(in_features=896, out_features=128, bias=True)
        (o_proj): Linear(in_features=896, out_features=896, bias=False)
        (rotary_emb): Qwen2RotaryEmbedding()
      )
      (mlp): Qwen2MLP(
        (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
        (up_proj): Linear(in_features=896, out_features=4864, bias=False)
        (down_proj): Linear(in_features=4864, out_features=896, bias=False)
        (act_fn): SiLU()
      )
      (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
    )
  )
  (norm): Qwen2RMSNorm((896,), eps=1e-06)
  (rotary_emb): Qwen2RotaryEmbedding()
)

In [ ]:
full_model.base_model

Qwen2Model(
  (embed_tokens): Embedding(151936, 896)
  (layers): ModuleList(
    (0-23): 24 x Qwen2DecoderLayer(
      (self_attn): Qwen2SdpaAttention(
        (q_proj): Linear(in_features=896, out_features=896, bias=True)
        (k_proj): Linear(in_features=896, out_features=128, bias=True)
        (v_proj): Linear(in_features=896, out_features=128, bias=True)
        (o_proj): Linear(in_features=896, out_features=896, bias=False)
        (rotary_emb): Qwen2RotaryEmbedding()
      )
      (mlp): Qwen2MLP(
        (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
        (up_proj): Linear(in_features=896, out_features=4864, bias=False)
        (down_proj): Linear(in_features=4864, out_features=896, bias=False)
        (act_fn): SiLU()
      )
      (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
    )
  )
  (norm): Qwen2RMSNorm((896,), eps=1e-06)
  (rotary_emb): Qwen2RotaryEmbedding()
)

## 看看base_model與model有何不同?

model內部有: 一個base_model+輸出分類層。它被拼接為分類器，可以做兩個類別的分類

base_model仍舊是GPT自回歸模型

但是在記憶體，兩者是共享同一份base_model權重，model有多一個分類器的輸出層的部分

In [ ]:
full_model 

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((

In [ ]:
model

QwenForClassifier(
  (base_model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMS

In [ ]:
# 這與model是一樣的模型??
model.base_model

Qwen2Model(
  (embed_tokens): Embedding(151936, 896)
  (layers): ModuleList(
    (0-23): 24 x Qwen2DecoderLayer(
      (self_attn): Qwen2SdpaAttention(
        (q_proj): Linear(in_features=896, out_features=896, bias=True)
        (k_proj): Linear(in_features=896, out_features=128, bias=True)
        (v_proj): Linear(in_features=896, out_features=128, bias=True)
        (o_proj): Linear(in_features=896, out_features=896, bias=False)
        (rotary_emb): Qwen2RotaryEmbedding()
      )
      (mlp): Qwen2MLP(
        (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
        (up_proj): Linear(in_features=896, out_features=4864, bias=False)
        (down_proj): Linear(in_features=4864, out_features=896, bias=False)
        (act_fn): SiLU()
      )
      (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
    )
  )
  (norm): Qwen2RMSNorm((896,), eps=1e-06)
  (rotary_emb): Qwen2RotaryEmbedding()
)

## hidden_states

In [ ]:
# Create a sample input
text = "國立高雄科技大學"
inputs = tokenizer(text, return_tensors="pt").to(device)

# Get hidden states from the base model
with torch.no_grad():
    outputs = model.base_model(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        output_hidden_states=True
    )
    
    hidden_states = outputs.hidden_states
    
    # Print dimensions info
    print(f"Number of layers in hidden_states: {len(hidden_states)}")
    print(f"First hidden state shape: {hidden_states[0].shape}")
    print(f"Last hidden state shape: {hidden_states[-1].shape}")
    
    # Print shape of each layer
    for i, hs in enumerate(hidden_states):
        print(f"Layer {i} shape: {hs.shape}")

Number of layers in hidden_states: 25
First hidden state shape: torch.Size([1, 5, 896])
Last hidden state shape: torch.Size([1, 5, 896])
Layer 0 shape: torch.Size([1, 5, 896])
Layer 1 shape: torch.Size([1, 5, 896])
Layer 2 shape: torch.Size([1, 5, 896])
Layer 3 shape: torch.Size([1, 5, 896])
Layer 4 shape: torch.Size([1, 5, 896])
Layer 5 shape: torch.Size([1, 5, 896])
Layer 6 shape: torch.Size([1, 5, 896])
Layer 7 shape: torch.Size([1, 5, 896])
Layer 8 shape: torch.Size([1, 5, 896])
Layer 9 shape: torch.Size([1, 5, 896])
Layer 10 shape: torch.Size([1, 5, 896])
Layer 11 shape: torch.Size([1, 5, 896])
Layer 12 shape: torch.Size([1, 5, 896])
Layer 13 shape: torch.Size([1, 5, 896])
Layer 14 shape: torch.Size([1, 5, 896])
Layer 15 shape: torch.Size([1, 5, 896])
Layer 16 shape: torch.Size([1, 5, 896])
Layer 17 shape: torch.Size([1, 5, 896])
Layer 18 shape: torch.Size([1, 5, 896])
Layer 19 shape: torch.Size([1, 5, 896])
Layer 20 shape: torch.Size([1, 5, 896])
Layer 21 shape: torch.Size([1, 5,

In [ ]:
print(f"Hidden size from config: {model.config.hidden_size}")

Hidden size from config: 896


In [ ]:
inputs

{'input_ids': tensor([[ 99941,  79095, 111257,  99602, 108007]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1]], device='cuda:0')}

##  layer_weights

在實際訓練後，layer_weights 的值可能會從初始的均勻分佈 [0.25, 0.25, 0.25, 0.25] 演變為類似於 [0.1, 0.2, 0.3, 0.4] 的分佈，表明模型學習到了不同層次特徵的相對重要性。這正是此設計的核心優勢 - 讓模型自動決定哪些層的表示更有價值，而不是人為固定權重。

In [ ]:
# 訓練前 model.layer_weights的權重
print("訓練前:", model.layer_weights.detach().cpu().numpy())

訓練前: [0.25 0.25 0.25 0.25]


In [ ]:
# 訓練前 model.layer_weights的權重
print("訓練前:", model.layer_weights)

訓練前: Parameter containing:
tensor([0.2500, 0.2500, 0.2500, 0.2500], device='cuda:0', requires_grad=True)


In [ ]:
# 訓練前 model.layer_weights的權重
print("訓練前:", F.softmax(model.layer_weights, dim=0).detach().cpu().numpy())

訓練前: [0.25 0.25 0.25 0.25]


# 模型怎麼用?

尚未訓練的模型 (classifier的數據都是0)

In [ ]:

# Function to make predictions
def predict_news_category(text, model, tokenizer, device):
    # Tokenize the input text
    inputs = tokenizer(
        text,
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)
    
    # Get model predictions
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Extract logits and apply softmax to get probabilities
    # logits = outputs.logits
    logits = outputs["logits"]  # 取出 logits
    
    
    probabilities = torch.nn.functional.softmax(logits, dim=-1)
    
    # Get the predicted class (0: negative, 1: positive)
    predicted_class = torch.argmax(probabilities, dim=-1).item()
    
    # Get the class name using id_to_label
    predicted_label = id_to_label[predicted_class]
    
    # Get the confidence score
    confidence = probabilities[0][predicted_class].item()
    
    return {
        "text": text,
        "sentiment": predicted_label,
        "confidence": round(confidence,2),
        "probabilities": {
            id_to_label[i]: round(prob.item(), 2) for i, prob in enumerate(probabilities[0])
        }
    }


In [ ]:
text='''立法院今天發布人事通報，總務處長周傑調任參事室參事並代理總務處長，引發外界聯想立法院長韓國瑜出手進行人事調整。周傑今天受訪表示，正常調動，謝謝關心。
部分媒體報導，質疑立法院總務處長人事異動，是否與先前立法院立委辦公室傳出有裝潢費昂貴、不合理等情形，或是日前立法院司法及法制委員會有委員提議要更改格局，而周傑回應「建議要更動是否可在會期中研議規畫，休會時再改」，令國民黨立法院黨團總召傅崐萁不以為然有關。'''
predict_news_category(text, model, tokenizer, device)

{'text': '立法院今天發布人事通報，總務處長周傑調任參事室參事並代理總務處長，引發外界聯想立法院長韓國瑜出手進行人事調整。周傑今天受訪表示，正常調動，謝謝關心。\n部分媒體報導，質疑立法院總務處長人事異動，是否與先前立法院立委辦公室傳出有裝潢費昂貴、不合理等情形，或是日前立法院司法及法制委員會有委員提議要更改格局，而周傑回應「建議要更動是否可在會期中研議規畫，休會時再改」，令國民黨立法院黨團總召傅崐萁不以為然有關。',
 'sentiment': '生活',
 'confidence': 0.14,
 'probabilities': {'政治': 0.09,
  '科技': 0.07,
  '運動': 0.04,
  '證卷': 0.12,
  '產經': 0.09,
  '娛樂': 0.07,
  '生活': 0.14,
  '國際': 0.11,
  '社會': 0.09,
  '文化': 0.09,
  '兩岸': 0.08}}

In [ ]:
text = "這個產品品質差，服務更糟糕"
predict_news_category(text, model, tokenizer, device)

{'text': '這個產品品質差，服務更糟糕',
 'sentiment': '兩岸',
 'confidence': 0.16,
 'probabilities': {'政治': 0.06,
  '科技': 0.07,
  '運動': 0.06,
  '證卷': 0.08,
  '產經': 0.08,
  '娛樂': 0.08,
  '生活': 0.15,
  '國際': 0.13,
  '社會': 0.06,
  '文化': 0.07,
  '兩岸': 0.16}}

In [ ]:
text = "這家餐廳的食物美味，環境也很舒適"
predict_news_category(text, model, tokenizer, device)

{'text': '這家餐廳的食物美味，環境也很舒適',
 'sentiment': '產經',
 'confidence': 0.16,
 'probabilities': {'政治': 0.06,
  '科技': 0.05,
  '運動': 0.07,
  '證卷': 0.07,
  '產經': 0.16,
  '娛樂': 0.07,
  '生活': 0.08,
  '國際': 0.06,
  '社會': 0.13,
  '文化': 0.1,
  '兩岸': 0.13}}

## 模型每個參數層都是Trainable

In [ ]:
def print_model_details(model):
    print("\n" + "="*80)
    print("MODEL ARCHITECTURE WITH TRAINABILITY STATUS")
    print("="*80)
    
    trainable_params = 0
    non_trainable_params = 0
    
    # Print each layer with details
    for idx, (name, layer) in enumerate(model.named_modules(), 1):
        if not list(layer.named_children()):  # Only print leaf nodes (actual layers)
            param_count = sum(p.numel() for p in layer.parameters())
            status = "TRAINABLE" if any(p.requires_grad for p in layer.parameters()) else "NON-TRAINABLE"
            
            print(f"\nLayer #{idx:02d}")
            print(f"├─ Name: {name}")
            print(f"├─ Type: {layer.__class__.__name__}")
            print(f"├─ Details: {layer}")
            print(f"├─ Parameters: {param_count:,}")
            print(f"└─ Status: {status}")
            
            if status == "TRAINABLE":
                trainable_params += param_count
            else:
                non_trainable_params += param_count
    
    total_params = trainable_params + non_trainable_params
    trainable_percentage = (trainable_params / total_params) * 100 if total_params > 0 else 0
    
    print("\n" + "="*80)
    print(f"Total Trainable Parameters: {trainable_params:,}")
    print(f"Total Non-Trainable Parameters: {non_trainable_params:,}")
    print(f"Total Parameters: {total_params:,}")
    print(f"Trainable Parameters Percentage: {trainable_percentage:.2f}%")
    print("="*80)

In [ ]:
print_model_details(model)


MODEL ARCHITECTURE WITH TRAINABILITY STATUS

Layer #03
├─ Name: base_model.embed_tokens
├─ Type: Embedding
├─ Details: Embedding(151936, 896)
├─ Parameters: 136,134,656
└─ Status: NON-TRAINABLE

Layer #07
├─ Name: base_model.layers.0.self_attn.q_proj
├─ Type: Linear
├─ Details: Linear(in_features=896, out_features=896, bias=True)
├─ Parameters: 803,712
└─ Status: NON-TRAINABLE

Layer #08
├─ Name: base_model.layers.0.self_attn.k_proj
├─ Type: Linear
├─ Details: Linear(in_features=896, out_features=128, bias=True)
├─ Parameters: 114,816
└─ Status: NON-TRAINABLE

Layer #09
├─ Name: base_model.layers.0.self_attn.v_proj
├─ Type: Linear
├─ Details: Linear(in_features=896, out_features=128, bias=True)
├─ Parameters: 114,816
└─ Status: NON-TRAINABLE

Layer #10
├─ Name: base_model.layers.0.self_attn.o_proj
├─ Type: Linear
├─ Details: Linear(in_features=896, out_features=896, bias=False)
├─ Parameters: 802,816
└─ Status: NON-TRAINABLE

Layer #11
├─ Name: base_model.layers.0.self_attn.rotary_emb

In [ ]:
# 檢查原始參數是否可訓練
# print("Classifier weight requires_grad:", model.classifier.weight.requires_grad)

## 模型主體固定參數，只有極少數參數是Trainable

In [ ]:
#model.print_trainable_parameters()

In [ ]:
def print_trainable_parameters(model):
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    all_params = sum(p.numel() for p in model.parameters())
    print(f"trainable params: {trainable_params:,d} || all params: {all_params:,d} || trainable%: {100 * trainable_params / all_params:.2f}%")

In [ ]:
print_trainable_parameters(model)

trainable params: 2,675,855 || all params: 496,708,623 || trainable%: 0.54%


In [ ]:
print_model_details(model)


MODEL ARCHITECTURE WITH TRAINABILITY STATUS

Layer #03
├─ Name: base_model.embed_tokens
├─ Type: Embedding
├─ Details: Embedding(151936, 896)
├─ Parameters: 136,134,656
└─ Status: NON-TRAINABLE

Layer #07
├─ Name: base_model.layers.0.self_attn.q_proj
├─ Type: Linear
├─ Details: Linear(in_features=896, out_features=896, bias=True)
├─ Parameters: 803,712
└─ Status: NON-TRAINABLE

Layer #08
├─ Name: base_model.layers.0.self_attn.k_proj
├─ Type: Linear
├─ Details: Linear(in_features=896, out_features=128, bias=True)
├─ Parameters: 114,816
└─ Status: NON-TRAINABLE

Layer #09
├─ Name: base_model.layers.0.self_attn.v_proj
├─ Type: Linear
├─ Details: Linear(in_features=896, out_features=128, bias=True)
├─ Parameters: 114,816
└─ Status: NON-TRAINABLE

Layer #10
├─ Name: base_model.layers.0.self_attn.o_proj
├─ Type: Linear
├─ Details: Linear(in_features=896, out_features=896, bias=False)
├─ Parameters: 802,816
└─ Status: NON-TRAINABLE

Layer #11
├─ Name: base_model.layers.0.self_attn.rotary_emb

## Use EOS token as padding

In [ ]:
tokenizer.pad_token

'<|endoftext|>'

In [ ]:
tokenizer.eos_token

'<|im_end|>'

In [ ]:
model.pad_token_id = tokenizer.eos_token_id  # Use EOS token as padding

# Train

In [ ]:
      
class CustomTrainer(Trainer):
    def save_model(self, output_dir=None, _internal_call=False):
        """保存分類器權重和配置"""
        # 使用預設output_dir如果未指定
        output_dir = output_dir if output_dir is not None else self.args.output_dir
        os.makedirs(output_dir, exist_ok=True)
        
        # 保存分類器權重
        classifier_path = os.path.join(output_dir, "classifier_weights.pt")
        model_dict = {
            'classifier': self.model.classifier.state_dict(),
            'layer_weights': self.model.layer_weights,
            'q_proj': self.model.q_proj.state_dict(),
            'k_proj': self.model.k_proj.state_dict(),
            'v_proj': self.model.v_proj.state_dict(),
            'config': {
                'num_labels': self.model.config.num_labels,
                'hidden_size': self.model.config.hidden_size
            }
        }
        # Save model config (required by HF Trainer)
        if hasattr(self.model, "config"):
            self.model.config.save_pretrained(output_dir)
        torch.save(model_dict, classifier_path)
        print(f"已保存分類器權重至 {classifier_path}")
        # 保存訓練狀態
        super().save_state()
        return output_dir


In [ ]:
#metric = evaluate.combine(["accuracy", "f1", "precision", "recall"])
#metric = evaluate.combine(["accuracy"])
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)  # Convert probabilities to predicted labels
    metric = evaluate.load('accuracy')
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
# requries several GB of GPU memory
training_args = TrainingArguments(
    output_dir="checkpoints_v1",  # Output directory for checkpoints
    learning_rate=5e-5,  # Learning rate for the optimizer
    weight_decay=0.01,  # Weight decay for regularization
    warmup_steps=500,
    seed=101,
    
    per_device_train_batch_size=26,  # Batch size per device
    per_device_eval_batch_size=3,  # Batch size per device for evaluation 
    
    num_train_epochs=5,  # Number of training epochs
    
    eval_strategy='steps',  # Evaluate after each epoch
    save_strategy="steps",  # Save model checkpoints after each epoch
    
    #load_best_model_at_end=True,  # Load the best model based on the chosen metric
    save_total_limit=2,
    push_to_hub=False,  # Disable pushing the model to the Hugging Face Hub 
    report_to="none",  # Disable logging to Weight&Bias
    #fp16=True, # 是否用此精度訓練Whether to use fp16 16-bit (mixed) precision training instead of 32-bit training.
    #bf16=True, # for 新型GPU 才能設定bf16
    logging_steps=500,
    save_steps=500,
)  

In [ ]:
trainer = CustomTrainer(
    model=model,  # The LoRA-adapted model
    #tokenizer=tokenizer,  # Tokenizer for the model
    processing_class=tokenizer,  # Tokenizer for the model
    
    train_dataset=train_data,  # Training dataset
    eval_dataset=test_data,  # Evaluation dataset
    args=training_args,  # Training arguments
    compute_metrics=compute_metrics,  # Function to calculate evaluation metrics
)

## Resume training from the last checkpoint if available

接續訓練

訓練後，手動載入之前訓練的分類器權重

In [ ]:

# checkpoint_path = "./checkpoints_v3/checkpoint-3167"
# model_path = "trained_classifier_v3"
# model.load_model(checkpoint_path)

## Let's train the model

In [ ]:
%%time
model.config.use_cache = False  # Disable cache for faster training
trainer.train()
# trainer.train(resume_from_checkpoint=True) # Resume training from a checkpoint
#trainer.train(resume_from_checkpoint="./checkpoints_v1/checkpoint-19002")

  0%|          | 0/4785 [00:00<?, ?it/s]

{'loss': 1.415, 'grad_norm': 11.18798542022705, 'learning_rate': 5e-05, 'epoch': 0.52}


  0%|          | 0/437 [00:00<?, ?it/s]

{'eval_loss': 0.6680538654327393, 'eval_accuracy': 0.826585179526356, 'eval_runtime': 20.9642, 'eval_samples_per_second': 62.44, 'eval_steps_per_second': 20.845, 'epoch': 0.52}
已保存分類器權重至 checkpoints_v1\checkpoint-500\classifier_weights.pt
{'loss': 0.5491, 'grad_norm': 3.8136565685272217, 'learning_rate': 4.41656942823804e-05, 'epoch': 1.04}


  0%|          | 0/437 [00:00<?, ?it/s]

...

  0%|          | 0/437 [00:00<?, ?it/s]

{'eval_loss': 0.2954573929309845, 'eval_accuracy': 0.9052711993888465, 'eval_runtime': 21.4832, 'eval_samples_per_second': 60.931, 'eval_steps_per_second': 20.341, 'epoch': 4.18}
已保存分類器權重至 checkpoints_v1\checkpoint-4000\classifier_weights.pt
{'loss': 0.2163, 'grad_norm': 2.620363235473633, 'learning_rate': 3.325554259043174e-06, 'epoch': 4.7}


  0%|          | 0/437 [00:00<?, ?it/s]

{'eval_loss': 0.30339542031288147, 'eval_accuracy': 0.9052711993888465, 'eval_runtime': 19.7239, 'eval_samples_per_second': 66.366, 'eval_steps_per_second': 22.156, 'epoch': 4.7}
已保存分類器權重至 checkpoints_v1\checkpoint-4500\classifier_weights.pt
已保存分類器權重至 checkpoints_v1\checkpoint-4785\classifier_weights.pt
{'train_runtime': 1806.5446, 'train_samples_per_second': 68.833, 'train_steps_per_second': 2.649, 'train_loss': 0.4254943098269535, 'epoch': 5.0}
CPU times: total: 29min 7s
Wall time: 30min 6s


TrainOutput(global_step=4785, training_loss=0.4254943098269535, metrics={'train_runtime': 1806.5446, 'train_samples_per_second': 68.833, 'train_steps_per_second': 2.649, 'total_flos': 0.0, 'train_loss': 0.4254943098269535, 'epoch': 5.0})

# Make a test

In [ ]:
text='''立法院今天發布人事通報，總務處長周傑調任參事室參事並代理總務處長，引發外界聯想立法院長韓國瑜出手進行人事調整。周傑今天受訪表示，正常調動，謝謝關心。
部分媒體報導，質疑立法院總務處長人事異動，是否與先前立法院立委辦公室傳出有裝潢費昂貴、不合理等情形，或是日前立法院司法及法制委員會有委員提議要更改格局，而周傑回應「建議要更動是否可在會期中研議規畫，休會時再改」，令國民黨立法院黨團總召傅崐萁不以為然有關。'''
predict_news_category(text, model, tokenizer, device)

{'text': '立法院今天發布人事通報，總務處長周傑調任參事室參事並代理總務處長，引發外界聯想立法院長韓國瑜出手進行人事調整。周傑今天受訪表示，正常調動，謝謝關心。\n部分媒體報導，質疑立法院總務處長人事異動，是否與先前立法院立委辦公室傳出有裝潢費昂貴、不合理等情形，或是日前立法院司法及法制委員會有委員提議要更改格局，而周傑回應「建議要更動是否可在會期中研議規畫，休會時再改」，令國民黨立法院黨團總召傅崐萁不以為然有關。',
 'sentiment': '政治',
 'confidence': 0.96,
 'probabilities': {'政治': 0.96,
  '科技': 0.0,
  '運動': 0.0,
  '證卷': 0.0,
  '產經': 0.0,
  '娛樂': 0.0,
  '生活': 0.02,
  '國際': 0.0,
  '社會': 0.01,
  '文化': 0.0,
  '兩岸': 0.0}}

In [ ]:
text = "這個產品品質差，服務更糟糕"
predict_news_category(text, model, tokenizer, device)

{'text': '這個產品品質差，服務更糟糕',
 'sentiment': '生活',
 'confidence': 0.6,
 'probabilities': {'政治': 0.02,
  '科技': 0.03,
  '運動': 0.01,
  '證卷': 0.02,
  '產經': 0.2,
  '娛樂': 0.01,
  '生活': 0.6,
  '國際': 0.02,
  '社會': 0.07,
  '文化': 0.02,
  '兩岸': 0.01}}

# Confusion Matrix分類混和矩陣

In [ ]:
def inference(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.softmax(outputs['logits'], dim=1)
    #probs = torch.softmax(outputs.logits, dim=1)
    return probs.argmax().item()

In [ ]:
print(test_data)

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 1309
})


In [ ]:
df_test_data = test_data.to_pandas()[['text','label']]

In [ ]:
df_test_data.head(2)

,text,label
0,副總統賴清德今天以總統特使身分，搭乘專機啟程出席28日宏都拉斯新任總統卡斯楚就職典禮，訂30...,0
1,高雄市府今天宣布，前鎮區化工廠COVID-19群聚今天新增4人確診，都是23日已匡列隔離對象...,6


In [ ]:
%%time
# df_test_data['prediction'] = df_test_data['text'].apply(lambda x: np.argmax(x, axis=0)) 

# 
df_test_data['prediction'] = df_test_data['text'].map(inference)

CPU times: total: 22.8 s
Wall time: 24.8 s


In [ ]:
df_test_data.head(2)

,text,label,prediction
0,副總統賴清德今天以總統特使身分，搭乘專機啟程出席28日宏都拉斯新任總統卡斯楚就職典禮，訂30...,0,0
1,高雄市府今天宣布，前鎮區化工廠COVID-19群聚今天新增4人確診，都是23日已匡列隔離對象...,6,6


In [ ]:
accuracy = (df_test_data['prediction'] == df_test_data['label']).mean()
print(f"Model Accuracy on Test Data: {accuracy:.4f}")

Model Accuracy on Test Data: 0.8946


In [ ]:
cm = metrics.confusion_matrix(df_test_data['label'], df_test_data['prediction'])
cm

array([[149,   0,   0,   1,   3,   0,   7,   3,   2,   3,   1],
       [  0,   5,   0,   0,   0,   0,   0,   3,   0,   0,   0],
       [  0,   0,  78,   0,   0,   0,   0,   1,   0,   0,   0],
       [  0,   0,   0,  71,   5,   0,   1,   1,   0,   0,   0],
       [  7,   1,   0,   4, 100,   0,   7,   2,   0,   0,   1],
       [  0,   0,   0,   0,   0,  26,   0,   4,   0,   2,   0],
       [  7,   0,  12,   0,   7,   2, 270,   4,   3,   1,   0],
       [  8,   0,   0,   0,   0,   0,   1, 284,   0,   1,   5],
       [  4,   0,   0,   0,   0,   0,  12,   1,  98,   0,   0],
       [  0,   0,   0,   0,   0,   2,   4,   1,   0,  26,   0],
       [  0,   0,   0,   0,   0,   1,   0,   3,   0,   0,  64]],
      dtype=int64)

In [ ]:
print(metrics.classification_report(df_test_data['label'], df_test_data['prediction']))

              precision    recall  f1-score   support

           0       0.85      0.88      0.87       169
           1       0.83      0.62      0.71         8
           2       0.87      0.99      0.92        79
           3       0.93      0.91      0.92        78
           4       0.87      0.82      0.84       122
           5       0.84      0.81      0.83        32
           6       0.89      0.88      0.89       306
           7       0.93      0.95      0.94       299
           8       0.95      0.85      0.90       115
           9       0.79      0.79      0.79        33
          10       0.90      0.94      0.92        68

    accuracy                           0.89      1309
   macro avg       0.88      0.86      0.87      1309
weighted avg       0.90      0.89      0.89      1309



# Save Model

In [ ]:
# 這樣存檔 - 只會儲存分類器權重和設定
model_path = "trained_classifier_v1"
model.save_model(model_path)
# model.save_pretrained(model_path)

已保存分類器權重至 trained_classifier_v1\classifier_weights.pt


# Load model

In [ ]:
# 在外部先載入base_model預訓練權重(不包含分類層)
# full_model = AutoModelForCausalLM.from_pretrained(model_id)
#
model_path = "trained_classifier_v1"
model_path = "checkpoints_v1/checkpoint-4785/"
hidden_size = full_model.config.hidden_size
model = QwenForClassifier(full_model.model, hidden_size, num_labels= len(categories))

model.load_model(model_path, device=device)

# 移動到指定設備
model = model.to(device)

已載入分類器權重: checkpoints_v1/checkpoint-4785/classifier_weights.pt


In [ ]:
text = "這個產品品質差，服務更糟糕"
predict_news_category(text, model, tokenizer, device)

{'text': '這個產品品質差，服務更糟糕',
 'sentiment': '生活',
 'confidence': 0.73,
 'probabilities': {'政治': 0.01,
  '科技': 0.02,
  '運動': 0.0,
  '證卷': 0.02,
  '產經': 0.1,
  '娛樂': 0.01,
  '生活': 0.73,
  '國際': 0.01,
  '社會': 0.08,
  '文化': 0.01,
  '兩岸': 0.01}}

In [ ]:
full_model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((

# Text generation fro full_model

In [ ]:
from IPython.display import Markdown

In [ ]:
def generate_text(input_prompt):
    
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": input_prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(device)

    prompt_length = model_inputs['input_ids'].shape[1]

    generated_ids = full_model.generate(
        model_inputs.input_ids,
        max_new_tokens=512,
        pad_token_id=tokenizer.eos_token_id
    )
    response = tokenizer.decode(generated_ids[0][prompt_length:], skip_special_tokens=True)
    return response


In [ ]:
text="給出三個保持健康的提示。"
result = generate_text(text)
Markdown(result)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


1. 適量運動：定期進行適度的運動可以改善心肺功能，提高免疫力和精神狀態。
2. 資源充足：充足的睡眠、均衡飲食以及充足的休息是維持健康的重要因素。
3. 保持良好的心理狀態：保持心理健康對健康起著關鍵作用，包括學習新技能、跟朋友聊天、參加社交活動等。

In [ ]:
%%time
text="我們如何減少空氣污染？請給幾項重要的建議。"
result = generate_text(text)
Markdown(result)

CPU times: total: 3.12 s
Wall time: 3.21 s


1. 出行方式：盡量選擇公共交通工具，如車、船、飛機等，而不是私家車。
2. 槐樹：槐樹可以吸收二氧化碳和一氧化碳，并且可以降低城市的温度，所以槐樹在城市中非常受欢迎。
3. 燃氣：使用低排放的能源，例如電力或風能，而不是煤炭或其他化石燃料。
4. 車輛排放：使用新能源汽车或者燃油效率更高的汽油车代替传统汽油车。
5. 保護植被：不要在森林中燃烧垃圾、丢弃塑料或其他废物，以减少空气污染。
6. 防止泄漏：如果你的汽车有漏气问题，你需要定期检查并维修它，以避免空气污染。
7. 支持环保政策：通过投票支持环保政策来推动政府采取更严格的措施保护环境。